**Before starting:** Runtime > Change runtime type > **T4 GPU** > Save.
Then run each cell with its play button, one at a time, top to bottom. Wait for the green tick before the next one.

In [ ]:
from google.colab import files
up = files.upload()

In [ ]:
!rm -rf mh_project && mkdir mh_project
!unzip -oq mh_project_step3.zip -d mh_project
%cd mh_project
!pip -q install shap statsmodels transformers datasets accelerate
!ls

In [ ]:
!python src/01_eda.py
!python src/02_train_classical.py

## 3. Train DistilBERT 
This takes a small pretrained language model and trains it a little more on our posts (*fine-tuning*).

In [ ]:
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)

print("GPU available:", torch.cuda.is_available())   # should say True
SEED = 42
set_seed(SEED)

def build(df):
    out = pd.DataFrame()
    out["raw_text"] = (df["title"].fillna("") + ". " + df["selftext"].fillna("")).str.strip()
    out["label"] = df["is_suicide"].astype(int)
    return out

train_full = build(pd.read_csv("data/training-set.csv"))
test = build(pd.read_csv("data/testing-set.csv")).reset_index(drop=True)
train, val = train_test_split(train_full, test_size=0.15, stratify=train_full["label"], random_state=SEED)
print(len(train), len(val), len(test))   # must print 1288 228 379

In [ ]:
MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(df):
    ds = Dataset.from_dict({"text": df["raw_text"].tolist(), "label": df["label"].tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True)

ds_train, ds_val, ds_test = to_ds(train), to_ds(val), to_ds(test)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def metrics(p):
    preds = p.predictions.argmax(-1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="binary")
    return {"accuracy": accuracy_score(p.label_ids, preds), "precision": pr, "recall": rc, "f1": f1}

args = TrainingArguments(
    output_dir="distilbert_out", num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1", seed=SEED, report_to="none", fp16=torch.cuda.is_available())

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  processing_class=tok, compute_metrics=metrics)
trainer.train()

In [ ]:
out = trainer.predict(ds_test)
logits = out.predictions
prob = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
pred = logits.argmax(-1)
y = test["label"].values
pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="binary")
print(f"DistilBERT TEST  accuracy={accuracy_score(y, pred):.4f}  precision={pr:.4f}  recall={rc:.4f}  "
      f"F1={f1:.4f}  ROC-AUC={roc_auc_score(y, prob):.4f}")
pd.DataFrame({"DistilBERT": pred, "DistilBERT_prob": prob, "y_true": y}).to_csv(
    "outputs/distilbert_test_predictions.csv", index=False)
print("saved outputs/distilbert_test_predictions.csv")

## 4. Compare ALL models, significance tests, errors, SHAP (3-5 minutes)

In [ ]:
!python src/03_evaluate_xai.py

## 5. Figures for your report

In [ ]:
from IPython.display import Image, display
for f in ["class_balance","length_before_after","top_words","confusion_matrices","roc_curves","shap_global",
          "shap_local_suicide_example","shap_local_depression_example"]:
    print(f); display(Image(f"outputs/figures/{f}.png"))

## 6. Download everything (tables + figures)

In [ ]:
!zip -qr results_to_download.zip outputs
files.download("results_to_download.zip")